# Step 1 · Economists from RePEc's rankings

**What this notebook does**

1. Downloads RePEc's two rankings of top economists:
   - [Publications in all years](https://ideas.repec.org/top/top.person.all.html)
   - [Publications in the last 10 years](https://ideas.repec.org/top/top.person.all10.html)
2. Reads every economist on them: their RePEc ID, name, main institution, whether they have died, and their
   rank on each list.
3. Makes one list of everyone who is on **either** ranking, and saves it as `repec_people.csv`.

Their homepages are collected in the next step, `02_repec_homepages.ipynb`.

**About RePEc.** RePEc (Research Papers in Economics) is the largest free database of economics research.
About 74,000 economists have a profile there. Each ranking lists the top 10% of them.

It takes a few seconds.

## Setup

The libraries this notebook uses:
- `requests` downloads web pages.
- `BeautifulSoup` (from the `bs4` package) reads a page's HTML so we can pick out names and links.
- `pandas` holds the results as a table and saves it as a CSV file.

In [1]:
import warnings

# The Mac's built-in Python prints a harmless warning about its SSL library when `requests` loads.
# Hide it. (This line has to come before `import requests`.)
warnings.filterwarnings("ignore")

import pandas as pd
import requests
from bs4 import BeautifulSoup

# Every request says who is asking. Some websites refuse requests that don't.
HEADERS = {"User-Agent": "Mozilla/5.0 (compatible; EconNobel poll; https://panahovf.github.io/econnobel/)"}

## 1. Download the two rankings

Each ranking is one long web page.

In [2]:
# The two rankings, with a short name for each.
RANKINGS = {
    "all_years": "https://ideas.repec.org/top/top.person.all.html",
    "last_10_years": "https://ideas.repec.org/top/top.person.all10.html",
}

ranking_pages = {}  # will hold the page text of each ranking, e.g. ranking_pages["all_years"]

for ranking_name, url in RANKINGS.items():
    response = requests.get(url, headers=HEADERS, timeout=60)
    response.raise_for_status()      # stop here if the download failed
    response.encoding = "utf-8"      # RePEc pages are UTF-8 (this keeps accents like é correct)
    ranking_pages[ranking_name] = response.text
    print(ranking_name, ": downloaded")

all_years : downloaded
last_10_years : downloaded


## 2. Read every economist from the rankings

Each economist is one row of a table on the page. The row has three cells:

| cell | what it holds | example |
|---|---|---|
| 1 | rank | `188` |
| 2 | the name (a link to their RePEc profile), a † if they have died, and their institutions underneath | `Susan Athey` / `Stanford University → Graduate School of Business, Stanford, …` |
| 3 | RePEc's score | `270.47` |

We go through every row and keep: the RePEc ID, the name, the main institution, whether they have died,
the profile address, and the rank.

In [3]:
rows = []  # one entry per economist per ranking

for ranking_name, page_text in ranking_pages.items():
    page = BeautifulSoup(page_text, "html.parser")

    for table_row in page.find_all("tr"):
        cells = table_row.find_all("td")
        if len(cells) < 3:
            continue  # not an economist (for example, the table's header row)

        # Economist rows start with their rank, a number. The page also has a small summary table whose
        # rows start with words ("Authors", "Institutions", ...): skip those.
        rank_text = cells[0].get_text(strip=True)
        if not rank_text.isdigit():
            continue

        # The first link in the second cell is the person's profile. Profiles live under /e/ or /f/.
        profile_link = cells[1].find("a", href=True)
        if profile_link is None:
            continue
        profile_url = profile_link["href"]                            # https://ideas.repec.org/e/pat6.html
        if "ideas.repec.org/e/" not in profile_url and "ideas.repec.org/f/" not in profile_url:
            continue
        repec_id = profile_url.split("/")[-1].replace(".html", "")    # pat6

        # The name. RePEc sometimes adds another spelling in brackets, "Jordi Galí (Jordi Gali)": keep the first.
        name = profile_link.get_text(strip=True).split(" (")[0]

        # RePEc puts a † after the name of an economist who has died.
        deceased = "†" in cells[1].get_text()

        # Institutions are listed under the name, main one first, e.g.
        # "Harvard University → Department of Economics, Cambridge, Massachusetts (United States)".
        # Keep the first one, and only the part before the arrow and before the first comma: "Harvard University".
        institution = ""
        first_affiliation = cells[1].find("small")
        if first_affiliation is not None:
            text = first_affiliation.get_text(" ", strip=True)
            text = text.split("→")[0]   # drop the department
            text = text.split(",")[0]   # drop the city and country
            institution = text.strip()

        rank = int(rank_text)

        rows.append({
            "repec_id": repec_id,
            "name": name,
            "institution": institution,
            "deceased": deceased,
            "profile_url": profile_url,
            "ranking": ranking_name,
            "rank": rank,
        })

rankings = pd.DataFrame(rows)

# How many economists each ranking has:
rankings["ranking"].value_counts()

ranking
last_10_years    3708
all_years        3707
Name: count, dtype: int64

## 3. One row per economist

About 1,800 economists are on both rankings. We keep one row for each person and add two columns with
their rank on each list (empty if they are not on that list).

In [4]:
# One row per person (if someone is on both lists, the first row found is kept).
people = rankings.drop_duplicates("repec_id")[["repec_id", "name", "institution", "profile_url"]].copy()

# Their rank on each list. .map() looks up each person's ID in the other table.
all_years = rankings[rankings["ranking"] == "all_years"].set_index("repec_id")["rank"]
last_10_years = rankings[rankings["ranking"] == "last_10_years"].set_index("repec_id")["rank"]
people["rank_all_years"] = people["repec_id"].map(all_years).astype("Int64")          # Int64 allows empty cells
people["rank_last_10_years"] = people["repec_id"].map(last_10_years).astype("Int64")

# Died: marked with a † on either list.
deceased_ids = set(rankings.loc[rankings["deceased"], "repec_id"])
people["deceased"] = people["repec_id"].isin(deceased_ids)

print(len(people), "economists in total")
print(people["rank_all_years"].notna().sum(), "on the all-years ranking")
print(people["rank_last_10_years"].notna().sum(), "on the last-10-years ranking")
print(people["deceased"].sum(), "marked as deceased")

5644 economists in total
3707 on the all-years ranking
3708 on the last-10-years ranking
286 marked as deceased


## 4. Save

The table is saved as `repec_people.csv`, one row per economist. The next step (`02_repec_homepages.ipynb`)
reads it.

In [5]:
people = people[["repec_id", "name", "institution", "deceased",
                  "rank_all_years", "rank_last_10_years", "profile_url"]]
people.to_csv("repec_people.csv", index=False)

print("Saved", len(people), "economists to repec_people.csv")
people.head(10)

Saved 5644 economists to repec_people.csv


,repec_id,name,institution,deceased,rank_all_years,rank_last_10_years,profile_url
0,psh93,Andrei Shleifer,Harvard University,False,1,79,https://ideas.repec.org/e/psh93.html
1,pac16,Daron Acemoglu,Massachusetts Institute of Technology (MIT),False,2,1,https://ideas.repec.org/e/pac16.html
2,phe22,James J. Heckman,University of Chicago,False,3,71,https://ideas.repec.org/e/phe22.html
3,pst33,Joseph Stiglitz,,False,4,117,https://ideas.repec.org/e/pst33.html
4,pti33,Jean Tirole,Toulouse School of Economics (TSE),False,5,83,https://ideas.repec.org/e/pti33.html
5,pba251,Robert J. Barro,Harvard University,False,6,54,https://ideas.repec.org/f/pba251.html
6,pli176,John List,University of Chicago,False,7,13,https://ideas.repec.org/e/pli176.html
7,pph8,Peter C. B. Phillips,,False,8,96,https://ideas.repec.org/e/pph8.html
8,pfa110,"Eugene F. Fama, Sr.",University of Chicago,False,9,<NA>,https://ideas.repec.org/e/pfa110.html
9,pca271,David E. Card,,False,10,56,https://ideas.repec.org/f/pca271.html
